# Hazus wind damage functions

## Hazus vulnerability functions
The [FEMA Hazus](https://www.fema.gov/flood-maps/tools-resources/flood-map-products/hazus/user-technical-manuals) team have provided an extract of the damage functions used in the Hazus 6.1 Hurricane Model in an Excel file HazusWindDamFunctions_Hazus61.xlsx. This is a highly granular, albeit US-specific, set of damage functions. Hazus includes a sophisticated model of hurricane impact, capable of modelling the specifics of different assets. In the following, the motivation is to combine Hazus damage functions with wind models, such as the IRIS model tropical cyclone model, which can be used for estimating the impact of physical climate change for a portfolio of assets for example for, but certainly not limited to, [climate attribution use-cases](https://www.worldweatherattribution.org/yet-another-hurricane-wetter-windier-and-more-destructive-because-of-climate-change/). In [a recent climate attribution study](https://www.imperial.ac.uk/grantham/research/climate-science/modelling-tropical-cyclones/climate-change-attribution-hurricane-milton/), the authors combined the IRIS hazard model with a generic parametric damage model. Both types of damage functions, the specific and the generic, are useful and it is interesting to consider how the Hazus model relates to the parametric approach. 

## Defining wind speed
In their 2009 review paper, {cite:t}`vickery2009hurricane` split wind field modelling into three steps. In the first step, mean wind speed at gradient height, i.e. at a height where surface friction has negligible effect, is obtained (aka gradient wind speed); in the second step, the gradient wind speed is adjusted to a surface level value, and in the third step this surface level wind is adjusted for terrain and averaging time. It is therefore important to ensure that the definition of wind speed in the hazard model is clear because it could be:
1) Gradient wind speed;
2) Sustained surface wind speed, usually by convention the 1 minute average wind speed at 10 metre height for open terrain;
3) Peak gust wind speed adjusted to the local terrain. 

Put another way, wind speed requires the specification of:
- Height (physrisk default 10 m)
- Reference averaging period (physrisk default 1 minute)
- Surface roughness (physrisk default 'open terrain' / 0.03 m) 

In principle a high-resolution wind model needs no specification of surface roughness, it being understood that local conditions are already captured. However a 10 km or 1 km resolution model is unlikely to capture this. 

## Gust speed / surface roughness
Common choices of reference averaging period are 3 seconds (sometimes called 'gust' wind speed), 1 minute, 3 minutes or 1 hour. In physrisk, wind speed hazard indicators are 1 minute speed unless specified otherwise. As noted in the technical manual 4.4.7, "In Hazus, all damage and loss functions are given as a function of peak gust wind speed, not the one-minute wind speed." It is therefore necessary to convert the damage functions to match the indicators; moreover it is appropriate that damage functions are specified in terms of the peak gust wind speed since this drives the damage.

Conversion between 1 minute, open-terrain wind speed at 10 metres height to local terrain and 3 second gust speed involves two conversion factors which are both material (although offsetting). As an example, the Engineering Sciences Data Unit (ESDU, 1982) boundary transition model gives a gust factor of 1.28 for converting 1 minute to 3 second wind speed assuming a 0.03 m terrain roughness. Section 4.4.7 (see e.g. Figure 4-72) of the [Hazus 5.1 technical manual](https://www.fema.gov/sites/default/files/documents/fema_hazus-hurricane-model-technical-manual-5-1.pdf) deals with the impact of surface roughness on wind speed. It provides an example of its model: 3 second gust wind speed is reduced by 18% for suburban terrain (0.35 m) relative to the open terrain value. In this example it can be seen that the factors largely offset, which perhaps justifies why some models approximate the combined gust speed/surface roughness conversion as unity. Of note are also the World Meteorological Organization guidelines of {cite:t}`harper2010guidelines`. See Table 1.1; for example a gust factor of 1.49 applies when inferring 3 second gust speed from a 60 s reference period wind speed. The 'Durst curve' (from the original paper {cite:t}`durst1960wind`) is used by the ASCE7-05 standard. Review articles such as {cite:t}`holmes2014gust` give an overview of the standards.

The First Street Foundation {cite:t}`firststreet2023wind` documents an appealing approach based on American Society for Civil Engineers (ASCE) guidelines. This approach relates 3 second gust wind speed, $v(z)$, at height $z$ to the 'basic' wind speed, $v_b$, which is the 3 second gust wind speed 10 m wind speed in 'Exposure C' terrain (which is broadly open terrain with scattered obstructions). It presents the following model:

$$
\frac{v(z)}{v_{b}} = \sqrt{2.01} \left( \frac{z}{z_g} \right) ^{1/\alpha} \\
\alpha = c_1 z_0^{-0.133} \\
z_g = c_2 z_0^{0.125} \\
$$

The constants are defined in metres:

$$
c_1 = 5.56 \\
c_2 = 450 \\
$$

Such a model can be combined with the gust factor of 1.28 above to take account of roughness on a fine scale. 

The focus in this section is on vulnerability functions and we do not discuss the gust/terrain adjustment beyond noting that this is material and would require relatively high resolution adjustment factors. The implication for vulnerability functions is that these should be a function of local 3 second gust speed; it is rather for the hazard model to apply the appropriate conversion. Slightly confusingly, Hazus damage curves also depend on surface roughness, however as explained by Section 8 of the Hazus 5.1 technical manual, this relates rather to the impact of wind-driven missiles.

## Hazus vulnerability functions and parametric damage functions
Hazus defines a set of 39 Specific Building Types (SBTs) and defines damage functions within each type for a number of different combinations of Wind Building Characteristics (WBCs) and terrains. As mentioned above, the terrain dimension relates to wind-driven missiles and not to surface roughness conversion. The mapping of assets to a specific curve is one of the challenges addressed by the Hazus tool (there are around 31,000 curves for the combinations of types and terrains for structural damage, for example).

The parameters of the parametric model introduced above originate from {cite:t}`eberenz2020regional`, which specifies parameters for the model:

$$
d = \frac{v'^3}{1 + v'^3} \\
v' = \frac{\max(v - v_t, 0)}{v_h - v_t}
$$

Here $d$ is the damage, expressed as a fraction of the asset value, a function of wind speed, $v$. The parameters are $v_t$, the threshold wind value and $v_h$, the wind speed such that fractional damage is 0.5 (i.e. half asset value). The origin of the default parameters of $v_h = 74.7 \text{m}\text{s}^{−1}$, $v_t = 25.7 \text{m}\text{s}^{−1}$ are described in {cite:t}`sealy2017hurricane`. The authors took Hazus damage curves 'pertaining to houses with masonry walls'. They found, based on the threshold, $v_t$, that 'corresponding wind speed associated with 50 percent of loss ranged between 240 and 320 km/h (67 and 89 m/s) for single masonry units and between 225 and 290 km/h (63 and 81 m/s) for multi and business masonry units'. The value of 74.7 m/s corresponds to the mean of the two mid-points of these ranges, i.e., 280 and 258 km/h. We revisit that analysis below to verify this correspondence of the parameters to the 3 second gust speed.

## Mapping asset characteristics to Hazus vulnerability functions
It is desirable that vulnerability functions can be constructed from assets characterized by the Open Exposure Data (OED) standard.
https://github.com/OasisLMF/ODS_OpenExposureData/blob/develop/OpenExposureData
In particular, the following 'Loc' input fields are relevant (case changed from Pascal to snake-case):
- occupancy_code
- construction_code
- number_of_storeys

The priority of physrisk is first to cover the cases:
1) Asset-specifics known: OED occupancy_code, construction_code and number_of_storeys is known for the asset;
2) Sectorial model: asset-class, asset type and location are provided.

To reiterate, in the following, a particular aim is to address the case where only approximate building characteristics are known and also estimate the uncertainty around that. We construct a set of average curves for the 39 SBTs and create vulnerability configuration entries for these. For each SBT, the combination of Wind Building Characteristics (WBCs) used in the mapping is tracked. We then construct mappings for 1 and 2. In the case of 2, these are additional vulnerability configuration entries. In the case of 1, the logic that performs the mapping to specific building type is done in code.

## Identifying the config lines
The code below produces configuration lines with asset_identifier containing keys "hazus_wind_spec_build_type" and "damage_type". The two keys together form a unique identifier, e.g. ```hazus_wind_spec_build_type=MLRI,damage_type=structure```. This identifies the curve that is the average over all WBC curves and all terrain curves for the specific building type 'MLRI', considering structural damage to the property.

Valid values of damage_type are 'structure', 'contents' and 'downtime'.

In [ ]:
import numpy as np
import plotly.io
from plotly.subplots import make_subplots
from IPython.display import HTML
from physrisk.vulnerability_models.configuration.hazus_config_builders import (
    ConfigBuilderHazusWind,
)

plotly.io.renderers.default = "notebook"

In [ ]:
# the file is too large to include in the repo; we need to download from S3 before we start as a one-off:
builder = ConfigBuilderHazusWind()
builder.download_inputs()

In [ ]:
# the building of the configuration items is then done as follows:
builder = ConfigBuilderHazusWind()
config_items = builder.build_config()
print(f"{len(config_items)} vulnerability configuration items created")
print(config_items[38])

117 vulnerability configuration items created
hazard_class='Wind' asset_class='Asset' asset_identifier='hazus_wind_spec_build_type=MH76HUD,damage_type=structure' indicator_id='max_speed' indicator_units='m/s' impact_id='damage' impact_units=None curve_type='indicator/piecewise_linear' points_x=[22.35, 24.59, 26.82, 29.06, 31.29, 33.53, 35.76, 38.0, 40.23, 42.47, 44.7, 46.94, 49.17, 51.41, 53.64, 55.88, 58.12, 60.35, 62.59, 64.82, 67.06, 69.29, 71.53, 73.76, 76.0, 78.23, 80.47, 82.7, 84.94, 87.17, 89.41, 91.64, 93.88, 96.11, 98.35, 100.58, 102.82, 105.05, 107.29, 109.52, 111.76] points_y=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.01, 0.01, 0.01, 0.02, 0.03, 0.05, 0.09, 0.13, 0.18, 0.25, 0.33, 0.4, 0.48, 0.55, 0.63, 0.7, 0.76, 0.81, 0.85, 0.89, 0.92, 0.94, 0.96, 0.97, 0.98, 0.98, 0.99, 0.99, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0] points_z=None points_kind=None cap_of_points_x=None cap_of_points_y=None activation_of_points_x=None baseline_quantile_of_points_x=None reference=None license=None


In [ ]:
# how to extract key values from the comma-separated asset_identifiers:
def asset_ids_to_keys(id_str: str):
    keys = {}
    for key_value in id_str.split(","):
        key, value = key_value.split("=")
        keys[key] = value
    return keys

In [ ]:
# here we plot out the masonry specific building type curves, each averaged over all terrain types:

fig1 = make_subplots(rows=1, cols=1)

masonry = [
    "MSF1",
    "MSF2",
    "MMUH1",
    "MMUH2",
    "MMUH3",
    "MLRM1",
    "MLRM2",
    "MLRI",
    "MERBL",
    "MERBM",
    "MERBH",
    "MECBL",
    "MECBM",
    "MECBH",
]

config_subset = [
    item
    for item in config_items
    if "damage_type=structure" in item.asset_identifier
    and any("hazus_wind_spec_build_type=" + m in item.asset_identifier for m in masonry)
]
for item in config_subset:
    fig1.add_scatter(
        x=item.points_x, y=item.points_y, row=1, col=1, name=item.asset_identifier
    )
fig1.update_xaxes(title="Wind speed (m/s)", title_font={"size": 14}, row=1, col=1)
fig1.update_yaxes(
    title="Damage as fraction of TIV", title_font={"size": 14}, row=1, col=1
)
fig1.update_layout(legend=dict(orientation="h", y=-0.1))
fig1.update_layout(margin=dict(l=20, r=20, t=20, b=20))
fig1.update_layout(
    height=800,
)


def vul(v, v_half):
    v_thresh = 25.7
    vn = np.where(v > v_thresh, v - v_thresh, 0) / (v_half - v_thresh)
    f = vn**3 / (1 + vn**3)
    return f


fig1.add_scatter(
    x=item.points_x, y=vul(np.array(item.points_x), 74.7), row=1, col=1, name="Eberenz"
)

# Render explicitly with include_mathjax=False: the default "notebook" renderer
# always embeds its own (legacy v2) MathJax, which conflicts with the MathJax
# already loaded by Sphinx for the $...$ math elsewhere on this page.
HTML(
    fig1.to_html(
        div_id="wind-hazus-masonry-curves",
        include_plotlyjs="cdn",
        include_mathjax=False,
        full_html=False,
    )
)